# AlignForge English — Stage 1: Base Language Model Pre-training (C0)

This notebook implements the foundational causal language model from scratch:
1. **Byte-level BPE Tokenizer** (vocabulary 2048, 100% UTF-8 roundtrip coverage).
2. **Causal Transformer Decoder** (~3.67M params, RMSNorm, RoPE, Causal Attention, GELU FFN).
3. **Pretrainer Loop** on TinyStories corpus to produce the base checkpoint `c0_base.pt`.

> **Note:** All code is 100% self-contained directly within notebook cells.

In [ ]:
# Environment Setup
# On Google Colab, install datasets and pyyaml if not already available:
# !pip install -q datasets pyyaml tqdm

import math
import os
from pathlib import Path
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

## 1. Custom Byte-Level BPE Tokenizer (1:1 from src)

In [ ]:
import json
from pathlib import Path
import re
from typing import Dict, List, Optional, Tuple, Union


# Pre-tokenization regex pattern (splitting on contractions, words, numbers, punctuation, spaces)
SPLIT_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?[A-Za-z]+| ?[0-9]+| ?[^\s\w]+|\s+"""
)


class ByteLevelBPETokenizer(ITokenizer):

    SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]

    def __init__(self) -> None:
        self._special_tokens = list(self.SPECIAL_TOKENS)
        self._pad_token_id = 0
        self._unk_token_id = 1
        self._bos_token_id = 2
        self._eos_token_id = 3

        # Encapsulated state dictionaries
        self._vocab: Dict[bytes, int] = {}
        self._inverse_vocab: Dict[int, bytes] = {}
        self._merges: Dict[Tuple[bytes, bytes], bytes] = {}
        self._merge_ranks: Dict[Tuple[bytes, bytes], int] = {}

        self._init_base_vocab()

    def _init_base_vocab(self) -> None:
        self._vocab.clear()
        self._inverse_vocab.clear()

        # 1. Special tokens
        for idx, token in enumerate(self._special_tokens):
            raw = token.encode("utf-8")
            self._vocab[raw] = idx
            self._inverse_vocab[idx] = raw

        # 2. 256 byte values
        offset = len(self._special_tokens)
        for b in range(256):
            raw = bytes([b])
            self._vocab[raw] = offset + b
            self._inverse_vocab[offset + b] = raw

    # Properties implementing ITokenizer and backward compatibility
    @property
    def vocab_size(self) -> int:
        return len(self._vocab)

    @property
    def pad_id(self) -> int:
        return self._pad_token_id

    @property
    def pad_token_id(self) -> int:
        return self._pad_token_id

    @property
    def unk_token_id(self) -> int:
        return self._unk_token_id

    @property
    def bos_id(self) -> int:
        return self._bos_token_id

    @property
    def bos_token_id(self) -> int:
        return self._bos_token_id

    @property
    def eos_id(self) -> int:
        return self._eos_token_id

    @property
    def eos_token_id(self) -> int:
        return self._eos_token_id

    @property
    def special_tokens(self) -> List[str]:
        return list(self._special_tokens)

    @special_tokens.setter
    def special_tokens(self, tokens: List[str]) -> None:
        self._special_tokens = list(tokens)

    @property
    def vocab(self) -> Dict[bytes, int]:
        return self._vocab

    @vocab.setter
    def vocab(self, val: Dict[bytes, int]) -> None:
        self._vocab = val

    @property
    def inverse_vocab(self) -> Dict[int, bytes]:
        return self._inverse_vocab

    @inverse_vocab.setter
    def inverse_vocab(self, val: Dict[int, bytes]) -> None:
        self._inverse_vocab = val

    @property
    def merges(self) -> Dict[Tuple[bytes, bytes], bytes]:
        return self._merges

    @property
    def merge_ranks(self) -> Dict[Tuple[bytes, bytes], int]:
        return self._merge_ranks

    def train(self, texts: List[str], target_vocab_size: int = 2048) -> None:
        self._init_base_vocab()
        self._merges.clear()
        self._merge_ranks.clear()

        # 1. Pre-tokenize and split words into sequences of 1-byte elements
        word_counts: Dict[Tuple[bytes, ...], int] = {}
        for text in texts:
            words = SPLIT_PATTERN.findall(text)
            for w in words:
                raw_bytes = w.encode("utf-8")
                token_tuple = tuple(bytes([b]) for b in raw_bytes)
                word_counts[token_tuple] = word_counts.get(token_tuple, 0) + 1

        num_merges = target_vocab_size - len(self._vocab)

        # 2. Iteratively merge most frequent adjacent byte pairs
        for rank in range(num_merges):
            pair_counts: Dict[Tuple[bytes, bytes], int] = {}
            for word_tokens, count in word_counts.items():
                for i in range(len(word_tokens) - 1):
                    pair = (word_tokens[i], word_tokens[i + 1])
                    pair_counts[pair] = pair_counts.get(pair, 0) + count

            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            merged = best_pair[0] + best_pair[1]

            new_token_id = len(self._vocab)
            self._vocab[merged] = new_token_id
            self._inverse_vocab[new_token_id] = merged
            self._merges[best_pair] = merged
            self._merge_ranks[best_pair] = rank

            # Apply merge to all words in training set
            new_word_counts: Dict[Tuple[bytes, ...], int] = {}
            for word_tokens, count in word_counts.items():
                new_tokens = self._merge_word(word_tokens, best_pair, merged)
                new_word_counts[new_tokens] = count
            word_counts = new_word_counts

    def _merge_word(
        self,
        tokens: Tuple[bytes, ...],
        pair: Tuple[bytes, bytes],
        replacement: bytes,
    ) -> Tuple[bytes, ...]:
        out: List[bytes] = []
        i = 0
        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):
                out.append(replacement)
                i += 2
            else:
                out.append(tokens[i])
                i += 1
        return tuple(out)

    def _bpe(self, word_bytes: bytes) -> List[bytes]:
        tokens = [bytes([b]) for b in word_bytes]
        if len(tokens) <= 1:
            return tokens

        while len(tokens) > 1:
            best_rank = float("inf")
            best_pair = None

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self._merge_ranks.get(pair)
                if rank is not None and rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            merged = self._merges[best_pair]
            tokens = list(self._merge_word(tuple(tokens), best_pair, merged))

        return tokens

    def encode(
        self,
        text: str,
        add_bos: bool = False,
        add_eos: bool = False,
    ) -> List[int]:
        words = SPLIT_PATTERN.findall(text)
        token_ids: List[int] = []

        if add_bos:
            token_ids.append(self._bos_token_id)

        for w in words:
            raw = w.encode("utf-8")
            bpe_tokens = self._bpe(raw)
            for tok in bpe_tokens:
                token_id = self._vocab.get(tok, self._unk_token_id)
                token_ids.append(token_id)

        if add_eos:
            token_ids.append(self._eos_token_id)

        return token_ids

    def decode(
        self,
        ids: List[int],
        skip_special_tokens: bool = True,
    ) -> str:
        byte_chunks: List[bytes] = []
        special_ids = {
            self._pad_token_id,
            self._unk_token_id,
            self._bos_token_id,
            self._eos_token_id,
        }

        for token_id in ids:
            if skip_special_tokens and token_id in special_ids:
                continue
            raw = self._inverse_vocab.get(token_id, b"")
            byte_chunks.append(raw)

        full_bytes = b"".join(byte_chunks)
        return full_bytes.decode("utf-8", errors="replace")

    def save(self, directory: Union[str, Path]) -> None:
        dir_path = Path(directory)
        dir_path.mkdir(parents=True, exist_ok=True)

        vocab_serializable = {
            tok.hex(): idx for tok, idx in self._vocab.items()
        }
        merges_serializable = [
            [p[0].hex(), p[1].hex()] for p in self._merges.keys()
        ]

        data = {
            "vocab": vocab_serializable,
            "merges": merges_serializable,
            "special_tokens": self._special_tokens,
        }

        with open(dir_path / "tokenizer.json", "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    @classmethod
    def load(cls, directory: Union[str, Path]) -> ByteLevelBPETokenizer:
        dir_path = Path(directory)
        with open(dir_path / "tokenizer.json", "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = cls()
        tok._special_tokens = data.get("special_tokens", cls.SPECIAL_TOKENS)
        tok._vocab = {bytes.fromhex(k): v for k, v in data["vocab"].items()}
        tok._inverse_vocab = {v: k for k, v in tok._vocab.items()}

        tok._merges.clear()
        tok._merge_ranks.clear()
        for rank, (p0, p1) in enumerate(data.get("merges", [])):
            pair = (bytes.fromhex(p0), bytes.fromhex(p1))
            merged = pair[0] + pair[1]
            tok._merges[pair] = merged
            tok._merge_ranks[pair] = rank

        return tok

## 2. Causal Transformer Decoder Architecture (1:1 from src)

In [ ]:
import math
from typing import Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):

    def __init__(self, dim: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        normed = x * torch.rsqrt(variance + self.eps)
        return normed * self.weight


class RotaryEmbedding(nn.Module):

    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0) -> None:
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base

        inv_freq = 1.0 / (self.base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int) -> None:
        t = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)  # (seq_len, dim // 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if seq_len > self.cos_cached.shape[0]:
            self._build_cache(seq_len)
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(
    q: torch.Tensor,
    k: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
    position_ids: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    cos = cos.unsqueeze(0).unsqueeze(1)  # (1, 1, seq_len, head_dim)
    sin = sin.unsqueeze(0).unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class CausalSelfAttention(nn.Module):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.dropout_p = dropout
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        # Handle dynamic KV Cache if present
        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        new_cache = (k, v) if use_cache else None
        total_k_len = k.shape[2]

        # Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Causal mask: only apply when decoding multiple tokens or uncached
        if T > 1:
            # Query positions: [total_k_len - T, total_k_len - 1]
            # Key positions: [0, total_k_len - 1]
            q_idx = torch.arange(total_k_len - T, total_k_len, device=x.device).unsqueeze(1)
            k_idx = torch.arange(total_k_len, device=x.device).unsqueeze(0)
            causal_mask = (k_idx > q_idx).unsqueeze(0).unsqueeze(0)  # (1, 1, T, total_k_len)
            scores = scores.masked_fill(causal_mask, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        context = torch.matmul(attn_weights, v)  # (B, n_heads, T, head_dim)
        context = context.transpose(1, 2).contiguous().view(B, T, C)

        out = self.resid_dropout(self.out_proj(context))
        return out, new_cache


class FeedForward(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff, bias=False)
        self.fc2 = nn.Linear(d_ff, d_model, bias=False)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.fc2(self.act(self.fc1(x))))


class TransformerBlock(nn.Module):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_ff: int,
        dropout: float = 0.1,
        norm_eps: float = 1e-5,
    ) -> None:
        super().__init__()
        self.attn_norm = RMSNorm(d_model, eps=norm_eps)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout=dropout)
        self.ffn_norm = RMSNorm(d_model, eps=norm_eps)
        self.ffn = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        # Pre-norm Self-Attention with residual connection
        normed_x = self.attn_norm(x)
        attn_out, new_cache = self.attn(
            normed_x, cos, sin, kv_cache=kv_cache, use_cache=use_cache
        )
        x = x + attn_out

        # Pre-norm FFN with residual connection
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_cache

from pathlib import Path
from typing import Any, List, Optional, Tuple, Union
import torch
import torch.nn as nn



class CausalTransformerModel(nn.Module, ICausalLM):

    def __init__(
        self,
        config: Optional[ModelArchitectureConfig] = None,
        vocab_size: int = 2048,
        d_model: int = 256,
        n_heads: int = 4,
        n_layers: int = 4,
        d_ff: int = 1024,
        max_seq_len: int = 2048,
        dropout: float = 0.1,
        norm_eps: float = 1e-5,
        rope_base: float = 10000.0,
    ) -> None:
        super().__init__()

        if config is not None:
            self._config = config
        else:
            self._config = ModelArchitectureConfig(
                d_model=d_model,
                n_heads=n_heads,
                n_layers=n_layers,
                d_ff=d_ff,
                vocab_size=vocab_size,
                max_seq_len=max_seq_len,
                dropout=dropout,
                eps=norm_eps,
            )

        self.vocab_size = self._config.vocab_size
        self.d_model = self._config.d_model
        self.n_heads = self._config.n_heads
        self.n_layers = self._config.n_layers
        self.d_ff = self._config.d_ff
        self.max_seq_len = self._config.max_seq_len

        # Token embedding
        self.tok_embeddings = nn.Embedding(self.vocab_size, self.d_model)
        self.drop = nn.Dropout(self._config.dropout)

        # RoPE provider
        self.rope = RotaryEmbedding(
            self.d_model // self.n_heads,
            max_seq_len=self.max_seq_len,
            base=rope_base,
        )

        # Transformer decoder blocks
        self.layers = nn.ModuleList([
            TransformerBlock(
                d_model=self.d_model,
                n_heads=self.n_heads,
                d_ff=self.d_ff,
                dropout=self._config.dropout,
                norm_eps=self._config.eps,
            )
            for _ in range(self.n_layers)
        ])

        # Final normalization & language model head
        self.norm = RMSNorm(self.d_model, eps=self._config.eps)
        self.lm_head = nn.Linear(self.d_model, self.vocab_size, bias=False)

        # Weight tying between token embeddings and LM head
        self.tok_embeddings.weight = self.lm_head.weight

        self._init_weights()

    @property
    def config(self) -> ModelArchitectureConfig:
        return self._config

    def _init_weights(self) -> None:
        for name, param in self.named_parameters():
            if "weight" in name and param.dim() >= 2:
                nn.init.normal_(param, mean=0.0, std=0.02)
            elif "bias" in name:
                nn.init.zeros_(param)

    def count_parameters(self) -> int:
        seen = set()
        total = 0
        for p in self.parameters():
            if p.requires_grad and p.data_ptr() not in seen:
                seen.add(p.data_ptr())
                total += p.numel()
        return total

    def get_num_params(self, non_embedding: bool = False) -> int:
        return self.count_parameters()

    def forward(
        self,
        input_ids: torch.Tensor,
        kv_caches: Optional[Union[list, IKVCache]] = None,
        use_cache: bool = False,
    ) -> Any:
        # Check if kv_caches is an IKVCache instance
        if isinstance(kv_caches, IKVCache):
            return self._forward_with_ikvcache(input_ids, kv_caches)

        B, T = input_ids.shape
        x = self.drop(self.tok_embeddings(input_ids))

        past_len = (
            kv_caches[0][0].shape[2]
            if (kv_caches is not None and kv_caches[0] is not None)
            else 0
        )
        total_len = past_len + T
        cos, sin = self.rope(x, total_len)
        cos = cos[past_len:total_len]
        sin = sin[past_len:total_len]

        new_caches = [] if use_cache else None

        for idx, layer in enumerate(self.layers):
            layer_cache = kv_caches[idx] if kv_caches is not None else None
            x, new_cache = layer(
                x, cos, sin, kv_cache=layer_cache, use_cache=use_cache
            )
            if use_cache:
                new_caches.append(new_cache)

        x = self.norm(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        return logits, new_caches

    def _forward_with_ikvcache(
        self, input_ids: torch.Tensor, cache: IKVCache
    ) -> torch.Tensor:
        B, T = input_ids.shape
        x = self.drop(self.tok_embeddings(input_ids))

        past_len = cache.current_seq_len
        total_len = past_len + T
        cos, sin = self.rope(x, total_len)
        cos = cos[past_len:total_len]
        sin = sin[past_len:total_len]

        for idx, layer in enumerate(self.layers):
            # Compute Q, K, V for this layer
            normed_x = layer.attn_norm(x)
            q = layer.attn.q_proj(normed_x).view(B, T, self.n_heads, layer.attn.head_dim).transpose(1, 2)
            k = layer.attn.k_proj(normed_x).view(B, T, self.n_heads, layer.attn.head_dim).transpose(1, 2)
            v = layer.attn.v_proj(normed_x).view(B, T, self.n_heads, layer.attn.head_dim).transpose(1, 2)

            q, k = layer.attn.apply_rope(q, k, cos, sin) if hasattr(layer.attn, 'apply_rope') else (q, k)

            full_k, full_v = cache.update(idx, k, v)
            scores = torch.matmul(q, full_k.transpose(-2, -1)) / (layer.attn.head_dim ** 0.5)

            if T > 1:
                total_k_len = full_k.shape[2]
                q_idx = torch.arange(total_k_len - T, total_k_len, device=x.device).unsqueeze(1)
                k_idx = torch.arange(total_k_len, device=x.device).unsqueeze(0)
                causal_mask = (k_idx > q_idx).unsqueeze(0).unsqueeze(0)
                scores = scores.masked_fill(causal_mask, float("-inf"))

            attn_weights = torch.softmax(scores, dim=-1)
            context = torch.matmul(attn_weights, full_v)
            context = context.transpose(1, 2).contiguous().view(B, T, self.d_model)
            attn_out = layer.attn.out_proj(context)

            x = x + attn_out
            x = x + layer.ffn(layer.ffn_norm(x))

        x = self.norm(x)
        return self.lm_head(x)

    def save_weights(self, path: Union[str, Path]) -> None:
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), p)

    def load_weights(self, path: Union[str, Path]) -> None:
        self.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))


# Backward compatibility alias
AlignForgeModel = CausalTransformerModel

## 3. Pre-training Dataset and Trainer (1:1 from src)

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
import time
from typing import Any, Dict, Optional
import torch
import torch.nn as nn
from torch.utils.data import DataLoader



class BaseTrainer(ITrainer, ABC):

    def __init__(
        self,
        model: nn.Module,
        dataloader: DataLoader,
        lr: float = 3e-4,
        weight_decay: float = 0.01,
        max_grad_norm: float = 1.0,
        device: Optional[torch.device] = None,
        use_amp: bool = True,
    ) -> None:
        self._device = device or (
            torch.device("cuda" if torch.cuda.is_available() else "cpu")
        )
        self._model = model.to(self._device)
        self._dataloader = dataloader
        self._max_grad_norm = max_grad_norm
        self._lr = lr
        self._weight_decay = weight_decay

        trainable_params = [p for p in self._model.parameters() if p.requires_grad]
        self._optimizer = torch.optim.AdamW(
            trainable_params, lr=lr, weight_decay=weight_decay
        )

        # Mixed Precision support via CUDA AMP
        self._use_amp = use_amp and (self._device.type == "cuda")
        self._scaler = torch.cuda.amp.GradScaler(enabled=self._use_amp)

    @property
    def model(self) -> nn.Module:
        return self._model

    @property
    def device(self) -> torch.device:
        return self._device

    @property
    def optimizer(self) -> torch.optim.Optimizer:
        return self._optimizer

    @property
    def use_amp(self) -> bool:
        return self._use_amp

    def autocast_context(self):
        return torch.cuda.amp.autocast(enabled=self._use_amp, dtype=torch.float16)

    def backward_and_step(self, loss: torch.Tensor) -> None:
        if self._use_amp:
            self._scaler.scale(loss).backward()
            self._scaler.unscale_(self._optimizer)
            nn.utils.clip_grad_norm_(self._model.parameters(), self._max_grad_norm)
            self._scaler.step(self._optimizer)
            self._scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(self._model.parameters(), self._max_grad_norm)
            self._optimizer.step()

    @abstractmethod
    def train_step(self, batch: Any) -> float:
        pass

    def train(self, max_steps: int = 50, log_interval: int = 10) -> Dict[str, float]:
        self._model.train()
        step_count = 0
        final_loss = 0.0
        t0 = time.time()

        while step_count < max_steps:
            for batch in self._dataloader:
                loss_val = self.train_step(batch)
                step_count += 1
                final_loss = loss_val

                if step_count % log_interval == 0 or step_count == max_steps:
                    amp_tag = " (FP16)" if self._use_amp else ""
                    print(f"Step {step_count:4d}/{max_steps}{amp_tag} | Loss: {final_loss:.4f}")

                if step_count >= max_steps:
                    break

        elapsed = time.time() - t0
        return {
            "final_loss": final_loss,
            "steps": float(step_count),
            "elapsed_seconds": elapsed,
        }

from pathlib import Path
import time
from typing import Any, Dict, List, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import yaml



class TextDataset(Dataset):

    def __init__(self, token_ids: List[int], seq_len: int = 128) -> None:
        self.seq_len = seq_len
        num_chunks = len(token_ids) // (seq_len + 1)
        self.chunks: List[torch.Tensor] = []
        for i in range(num_chunks):
            chunk = token_ids[i * (seq_len + 1) : (i + 1) * (seq_len + 1)]
            self.chunks.append(torch.tensor(chunk, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.chunks)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk = self.chunks[idx]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y


class Pretrainer(BaseTrainer):

    def train_step(self, batch: Tuple[torch.Tensor, torch.Tensor]) -> float:
        x, y = batch
        x, y = x.to(self.device), y.to(self.device)

        self.optimizer.zero_grad()
        with self.autocast_context():
            out = self.model(x)
            logits = out[0] if isinstance(out, tuple) else out
            loss = F.cross_entropy(logits.view(-1, self.model.vocab_size), y.view(-1))

        self.backward_and_step(loss)
        return float(loss.item())


def train_pretrain(
    config_path: str | Path = "configs/model.yaml",
    max_steps: Optional[int] = None,
    output_checkpoint: str | Path = "artifacts/checkpoints/c0_base.pt",
    device_override: Optional[str] = None,
    batch_size: Optional[int] = None,
    use_amp: bool = True,
) -> Dict[str, float]:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    device = torch.device(
        device_override or ("cuda" if torch.cuda.is_available() else "cpu")
    )

    tok_dir = Path("artifacts/tokenizer")
    if (tok_dir / "tokenizer.json").exists():
        tokenizer = ByteLevelBPETokenizer.load(tok_dir)
    else:
        starter_stories = [
            "Once upon a time, there was a little girl named Lily. She loved to run in the park.",
            "Tim had a red car. He played with his car every day in his room.",
            "The sun was bright and warm. A little puppy barked happily in the green garden.",
            "A bird flew high into the blue sky. It sang a sweet song for all the animals.",
            "She helped her mother bake a cake. The cake smelled delicious and sweet.",
            "The cat sat on the soft mat. It purred softly as the boy petted its fur.",
        ] * 20
        tokenizer = ByteLevelBPETokenizer()
        tokenizer.train(starter_stories, target_vocab_size=cfg["model"]["vocab_size"])
        tok_dir.mkdir(parents=True, exist_ok=True)
        tokenizer.save(tok_dir)

    model = CausalTransformerModel(
        vocab_size=cfg["model"]["vocab_size"],
        d_model=cfg["model"]["d_model"],
        n_heads=cfg["model"]["n_heads"],
        n_layers=cfg["model"]["n_layers"],
        d_ff=cfg["model"]["d_ff"],
        max_seq_len=cfg["model"]["max_seq_len"],
        dropout=cfg["model"]["dropout"],
        norm_eps=float(cfg["model"]["norm_eps"]),
        rope_base=float(cfg["model"]["rope_base"]),
    ).to(device)

    corpus_text = (
        "Once upon a time, a little girl named Lily lived in a small house near the woods. "
        "Lily had a fluffy dog named Spot. Spot loved to run and jump. "
        "Every morning, Lily and Spot walked to school together. "
        "They saw birds singing in the trees and flowers blooming in the yard. "
        "One sunny afternoon, Spot found a shiny golden key under an old oak tree. "
        "Lily picked up the key and smiled. She wondered what magical door it would open."
    ) * 50
    tokens = tokenizer.encode(corpus_text, add_bos=True, add_eos=True)
    dataset = TextDataset(tokens, seq_len=cfg["model"]["max_seq_len"])
    bs = batch_size or (32 if device.type == "cuda" else cfg["training"]["batch_size"])
    loader = DataLoader(
        dataset,
        batch_size=bs,
        shuffle=True,
    )

    trainer = Pretrainer(
        model=model,
        dataloader=loader,
        lr=float(cfg["training"]["learning_rate"]),
        weight_decay=float(cfg["training"]["weight_decay"]),
        max_grad_norm=float(cfg["training"]["max_grad_norm"]),
        device=device,
        use_amp=use_amp,
    )

    steps = max_steps or 50
    metrics = trainer.train(max_steps=steps)

    out_path = Path(output_checkpoint)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": cfg,
            "step": steps,
            "loss": metrics["final_loss"],
            "vocab_size": model.vocab_size,
        },
        out_path,
    )

    return metrics

## 4. Run Pre-training Loop (C0 Base Checkpoint)

In [ ]:
# 1. Initialize tokenizer
tok_dir = Path('artifacts/tokenizer')
tok_dir.mkdir(parents=True, exist_ok=True)
tokenizer = ByteLevelBPETokenizer()
starter_corpus = [
    'Once upon a time, there was a little girl named Lily. She loved to run in the park.',
    'Tim had a small red car. He played with his car every day in his room.',
    'The sun was bright and warm. A little puppy barked happily in the green garden.',
    'A bird flew high into the blue sky. It sang a sweet song for all the animals.',
    'She helped her mother bake a cake. The cake smelled delicious and sweet.',
    'The cat sat on the soft mat. It purred softly as the boy petted its fur.',
] * 25
tokenizer.train(starter_corpus, target_vocab_size=2048)
tokenizer.save(tok_dir)
print(f'Trained BPE Tokenizer with vocabulary size: {tokenizer.vocab_size}')

# 2. Build model
model = CausalTransformerModel(
    vocab_size=2048,
    d_model=256,
    n_heads=4,
    n_layers=4,
    d_ff=1024,
    max_seq_len=256,
    dropout=0.1,
).to(device)
print(f'Model initialized with {model.count_parameters():,} trainable parameters.')

# 3. Prepare data loader
corpus_text = ' '.join(starter_corpus)
token_ids = tokenizer.encode(corpus_text, add_bos=True, add_eos=True)
dataset = TextDataset(token_ids, seq_len=128)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

# 4. Train model
trainer = Pretrainer(model=model, dataloader=loader, lr=3e-4, device=device)
metrics = trainer.train(max_steps=50, log_interval=10)

# 5. Save C0 Checkpoint
ckpt_dir = Path('artifacts/checkpoints')
ckpt_dir.mkdir(parents=True, exist_ok=True)
model.save_weights(ckpt_dir / 'c0_base.pt')
print(f'Base C0 model successfully trained and saved! Final loss: {metrics["final_loss"]:.4f}')